In [ ]:
import random
import sys
import numpy as np
import pandas as pd
import torch

from functools import partial
from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader
from transformers import RTDetrImageProcessor

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.data_prep import load_annotations
from scripts.dataset_helpers import DetectionDataset, RTDetrCollator
from scripts.model_builders import build_rtdetr
from scripts.experiment_runner import run_rtdetr

DIRECTORY = PROJECT_ROOT / "data"
INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"

W1106 23:11:34.196000 63434 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

SEED = 42
EPOCHS = 5
BATCH_SIZE = 2
FRAME_STEP = 15
CHECKPOINT = "PekingU/rtdetr_r50vd"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
processor = RTDetrImageProcessor.from_pretrained(CHECKPOINT)

coco_train = COCO(str(INFO_TRAIN))
coco_valid = COCO(str(INFO_VAL))
train_frames, train_boxes = load_annotations(coco_train, split="train")
valid_frames, valid_boxes = load_annotations(coco_valid, split="val")

train_annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
valid_annotations_by_image = {image_id: group for image_id, group in valid_boxes.groupby("image_id")}

sampled_train_frames = (train_frames.sort_values(["video", "frame_id"])
                        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

category_ids = sorted(coco_valid.getCatIds())
cat_to_label = {category_id: index for index, category_id in enumerate(category_ids)}
id_to_label = {index: coco_valid.cats[category_id]["name"] for category_id, index in cat_to_label.items()}
label_to_id = {name: index for index, name in id_to_label.items()}

loading annotations into memory...
Done (t=8.80s)
creating index...
index created!
loading annotations into memory...
Done (t=0.18s)
creating index...
index created!


KeyboardInterrupt: 

In [ ]:
train_dataset = DetectionDataset(frames=sampled_train_frames, directory=DIRECTORY,
    annotations_by_image=train_annotations_by_image,
    empty_annotations=train_boxes.iloc[:0].copy(), validation=False,
)
valid_dataset = DetectionDataset(
    frames=valid_frames, directory=DIRECTORY,
    annotations_by_image=valid_annotations_by_image,
    empty_annotations=valid_boxes.iloc[:0].copy(), validation=True,
)
collator = RTDetrCollator(processor, cat_to_label)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collator)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collator)

In [ ]:
model_kwargs = {"CHECKPOINT": CHECKPOINT, "id_to_label": id_to_label,
                "label_to_id": label_to_id, "num_queries": 150}

train_kwargs = {"processor": processor, "train_loader": train_loader,
                "valid_loader": valid_loader, "annotations_path": INFO_VAL,
                "device": device, "seed": SEED, "epochs": EPOCHS,
                "lr": 1e-4, "weight_decay": 1e-4, "max_hours": 3}

In [ ]:
baseline_histories, baseline_results = run_rtdetr(**train_kwargs, 
                                                  model_method=partial(build_rtdetr, **model_kwargs),
                                                  output_dir=PROJECT_ROOT / "outputs" / "rtdetr_baseline")

In [ ]:
box_histories, box_results = run_rtdetr(**train_kwargs,
                                        model_method=partial(build_rtdetr, **model_kwargs, matcher_bbox_cost=7.0),
                                        output_dir=PROJECT_ROOT / "outputs" / "rtdetr_bbox_cost_7")

In [ ]:
box_scheduler_histories, box_scheduler_results = run_rtdetr(**train_kwargs,
                                                            model_method=partial(build_rtdetr, **model_kwargs, matcher_bbox_cost=7.0),
                                                            output_dir=PROJECT_ROOT / "outputs" / "rtdetr_box_scheduler", lr_scheduler=True, step_size=4)